# NullVector Postgres Cookbook

Process a PDF into a searchable hierarchical tree with **4 lines of code** using `NullVectorClient`.

**Prerequisites:**
- PostgreSQL reachable via `NULLVECTOR_POSTGRES_CONNINFO` (default: `postgresql://REDACTED_DB_CRED@localhost:5432/app`)
- Groq API keys configured inline below (set `NULLVECTOR_NOTEBOOK_ENABLE_BRAIN=0` to disable)
- Project dependencies installed (`uv sync --extra dev`)

## What is NullVector?

NullVector is a **vectorless hierarchical RAG framework** — it processes PDFs and Markdown
into auditable, searchable hierarchical trees *without vector embeddings*. All retrieval is
structural (BM25-style ranking, Jaccard similarity, tree traversal) with full spatial
traceability back to source bounding boxes.

### Pipeline at a glance

```
PDF / Markdown
    │
    ▼
┌─────────────┐    Parse pages, extract text, detect outlines
│  Acquisition │    → CanonicalDocumentLedger (text substrate + page geometry)
└──────┬──────┘
       ▼
┌─────────────┐    Assemble heading hierarchy, verify coverage
│  Tree Build  │    → HierarchyNode tree (sections, subsections, page spans)
└──────┬──────┘
       ▼
┌─────────────┐    Merge small nodes for efficient serving
│  Compaction  │    → CompactedTreeNode[] (serving nodes → canonical nodes)
└──────┬──────┘
       ▼
┌─────────────┐    Create queryable retrieval units from tree nodes
│  Retrieval   │    → RetrievalCorpus (searchable text units with page refs)
│  Index       │
└──────┬──────┘
       ▼
┌─────────────┐    Structural search + BM25 ranking over the corpus
│  Search      │    → RetrievalHit[] (ranked results with page citations)
└──────┬──────┘
       ▼
┌─────────────┐    Grounded question answering with evidence
│  QA          │    → Answer with page-level citations
└─────────────┘
```

### Key concepts

| Concept | What it means |
|---------|---------------|
| **Acquisition** | Deterministic, CPU-only extraction of text, outlines, and page geometry from a PDF or Markdown file |
| **Hierarchy node** | A section in the document tree — has a title, page span, and owned text |
| **Compaction** | Merging small sibling nodes into larger "serving nodes" for efficient retrieval |
| **Retrieval unit** | A searchable text chunk derived from a tree node — the atomic unit that search operates over |
| **Tree search** | Navigating the hierarchy tree to find relevant sections, then ranking their retrieval units |
| **Tree summarization** | LLM-powered bottom-up summarization of tree nodes — enriches search with semantic context |
| **Grounded QA** | Answering questions using only evidence found in the corpus — never hallucinating |

### What this notebook covers

1. **Setup** — imports, PostgreSQL connection, Groq round-robin LLM gateway
2. **Ingest** — `client.ingest()` runs acquisition → tree build → retrieval in one call
3. **Summarize** — `client.build_tree(summarize=True)` re-builds the tree with LLM summaries
4. **Describe** — `client.build_description()` generates a document-level summary
5. **Search** — `client.search()` finds relevant sections with page citations
6. **QA** — `client.ask()` answers questions with three modes (summary, focused, low-evidence)

## Setup

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path


def banner(title: str) -> None:
    print()
    print("=" * 60)
    print(title)
    print("=" * 60)


def show_json(title: str, payload: object) -> None:
    banner(title)
    print(json.dumps(payload, indent=2, sort_keys=True, default=str))


ROOT = Path.cwd()
print(f"Repository root: {ROOT}")

Repository root: /home/pruthvi/projects/NullVector/cookbook


### Imports

Only three NullVector imports needed for the client path. The gateway and storage
configuration are the only framework internals you touch directly.

In [2]:
from nullvector import NullVectorClient
from nullvector.llm import GatewayAuditConfig, GatewayConfig, GatewayService
from nullvector.observability import (
    DEFAULT_OBSERVABILITY_JSONL_PATH,
    configure_default_runtime_observability,
)
from nullvector.storage import PostgresStorageConfig

try:
    import litellm

    from nullvector.llm.adapters import LiteLLMAdapter
except ImportError as exc:
    litellm = None
    LiteLLMAdapter = None
    LITELLM_IMPORT_ERROR = exc
else:
    LITELLM_IMPORT_ERROR = None

### Configuration

All paths, model names, and feature flags. Run IDs are **auto-generated by default** —
re-running creates fresh client-managed runs unless you pass explicit run IDs.

In [ ]:
COOKBOOK_RUNTIME_ROOT = ROOT / ".artifacts" / "cookbook" / "03_postgres_unified"
COOKBOOK_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)

# Source PDF — relative to the cookbook directory
PDF_SOURCE_PATH = str(ROOT / "903000608.pdf")

# PostgreSQL connection
POSTGRES_CONNINFO = os.environ.get(
    "NULLVECTOR_POSTGRES_CONNINFO",
    "postgresql://REDACTED_DB_CRED@localhost:5432/app",
)
POSTGRES_SCHEMA = os.environ.get("NULLVECTOR_POSTGRES_SCHEMA", "public")

# LLM model — defaults to Groq's Llama 4 Scout via LiteLLM
GROQ_DEFAULT_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"
DEFAULT_MODEL = os.environ.get("NULLVECTOR_LLM_MODEL", GROQ_DEFAULT_MODEL)
LITELLM_MODEL = DEFAULT_MODEL if DEFAULT_MODEL.startswith("groq/") else f"groq/{DEFAULT_MODEL}"

# Feature flags
ENABLE_BRAIN = os.environ.get("NULLVECTOR_NOTEBOOK_ENABLE_BRAIN", "1") != "0"

# Workspace root for NullVectorClient's local catalog
WORKSPACE = ROOT / ".artifacts" / "cookbook" / "03_workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)

### Groq Round-Robin

NullVector's LLM gateway owns retries internally. We rotate across 5 Groq API keys
to avoid per-key rate limits during notebook execution.

In [4]:
def parse_groq_api_keys(raw_keys: str) -> tuple[str, ...]:
    keys = tuple(part.strip() for part in raw_keys.split(",") if part.strip())
    if not keys:
        raise ValueError("GROQ_API_KEYS must contain at least one non-empty key.")
    return keys


def groq_key_label(slot: int, key: str) -> str:
    suffix = key[-4:] if len(key) >= 4 else key
    return f"slot-{slot:02d} (gsk_...{suffix})"


# 5 Groq API keys for round-robin rotation
GROQ_API_KEYS_INLINE = ",".join(
    (
        "REDACTED_GROQ_KEY",
        "REDACTED_GROQ_KEY",
        "REDACTED_GROQ_KEY",
        "REDACTED_GROQ_KEY",
        "REDACTED_GROQ_KEY",
    )
)
GROQ_API_KEYS = parse_groq_api_keys(GROQ_API_KEYS_INLINE)
GROQ_MASKED_KEYS = tuple(
    groq_key_label(slot, key) for slot, key in enumerate(GROQ_API_KEYS, start=1)
)

groq_key_cursor = 0


def next_groq_key_selection() -> tuple[str, str]:
    global groq_key_cursor
    index = groq_key_cursor % len(GROQ_API_KEYS)
    groq_key_cursor += 1
    return GROQ_API_KEYS[index], GROQ_MASKED_KEYS[index]

### Gateway & Client

Build the LLM gateway (live Groq round-robin when available, `None` otherwise).
When `gateway=None`, the pipeline runs fully deterministic — no LLM calls.
When a gateway is available, tree summarization and LLM-backed descriptions are enabled.

In [5]:
pg_config = PostgresStorageConfig(conninfo=POSTGRES_CONNINFO, schema=POSTGRES_SCHEMA)
OBSERVABILITY_JSONL_PATH = os.environ.get(
    "NULLVECTOR_OBSERVABILITY_JSONL_PATH",
    DEFAULT_OBSERVABILITY_JSONL_PATH,
)
runtime_logger = configure_default_runtime_observability(jsonl_path=OBSERVABILITY_JSONL_PATH)


def build_demo_gateway() -> tuple[GatewayService | None, str]:
    """Build a live Groq gateway or return None for deterministic-only mode."""
    audit_root = COOKBOOK_RUNTIME_ROOT / "gateway_audit"
    audit_root.mkdir(parents=True, exist_ok=True)
    if not ENABLE_BRAIN:
        return None, "disabled"
    if LiteLLMAdapter is not None and litellm is not None:
        def rotating_completion(**kwargs):
            api_key, label = next_groq_key_selection()
            print(f"Groq rotation -> {label}")
            kwargs["api_key"] = api_key
            return litellm.completion(**kwargs)

        return (
            GatewayService(
                GatewayConfig(
                    default_model=LITELLM_MODEL,
                    audit=GatewayAuditConfig(persist_root=str(audit_root)),
                ),
                provider_adapter=LiteLLMAdapter(completion_fn=rotating_completion),
                logger=runtime_logger,
            ),
            "litellm-groq-round-robin",
        )
    if LITELLM_IMPORT_ERROR is not None:
        print(f"LiteLLM unavailable: {LITELLM_IMPORT_ERROR}")
    # No gateway — pipeline runs fully deterministic (no LLM calls)
    return None, "none (deterministic-only)"


gateway, gateway_mode = build_demo_gateway()

client = NullVectorClient(
    WORKSPACE,
    storage=pg_config,
    gateway=gateway,
    logger=runtime_logger,
)

show_json("Client ready", {
    "workspace": str(WORKSPACE),
    "gateway_mode": gateway_mode,
    "postgres_schema": POSTGRES_SCHEMA,
    "groq_key_count": len(GROQ_API_KEYS),
})


Client ready
{
  "gateway_mode": "litellm-groq-round-robin",
  "groq_key_count": 5,
  "postgres_schema": "public",
  "workspace": "/home/pruthvi/projects/NullVector/cookbook/.artifacts/cookbook/03_workspace"
}


## Ingest — One Call Does Everything

`client.ingest()` runs the full pipeline in one call:
1. **Acquisition** — CPU-only PDF parsing (text, outlines, page geometry)
2. **Tree Build** — heading hierarchy assembly and verification
3. **Retrieval Index** — searchable corpus from tree nodes

No manual service construction, no manifest refs, no run ID wiring.

In [8]:
result = client.ingest(PDF_SOURCE_PATH)

show_json("Ingest complete", {
    "document_id": result.document_id,
    "acquisition_manifest": result.acquisition_manifest_path,
    "tree_manifest": result.tree_manifest_path,
    "retrieval_manifest": result.retrieval_manifest_path,
})

[SourceFingerprintComputed] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896
[AcquisitionStarted] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 run=903000608-acquisition provider=native_pymupdf


[HierarchyStrategySelected] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 tree=007 strategy=inferred_with_llm_assist
[NodeCommitted] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 tree=007 title=The Constitution of India node=b72cef9053fc86e4be077af1875741112039b775868e77c585c6ed8efdc9fcd1
[NodeCommitted] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 tree=007 title=Chapter IV A node=a64e79575e742e927a25cd80eab173bce22c936e163bd4c44f58f60e0b4137b4
[NodeCommitted] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 tree=007 title=Fundamental Duties node=0b3fce0ba956b638330d7d89234ba39f236c2bbfb286cf907e9e5e5e4563f484
[NodeCommitted] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 tree=007 title=Sachin Mehta node=a1147d47018b372a903d9586f5261aa50e6a079e2b4551a3c5f6259309b0f594
[NodeCommitted] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c58670


Ingest complete
{
  "acquisition_manifest": "pg://acquisition/903000608-acquisition/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/manifest.json",
  "document_id": "798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896",
  "retrieval_manifest": "pg://retrieval/903000608-retrieval/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/manifest.json",
  "tree_manifest": "pg://tree/007/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/manifest.json"
}


## Tree Summarization (LLM-Powered)

When a live gateway is available, `client.build_tree(summarize=True)` re-builds the
hierarchy tree with LLM-generated summaries for each node. This enriches search and QA
with semantic context beyond raw text.

When `gateway=None`, this step is a no-op — the tree stays deterministic.

In [ ]:
if gateway is not None:
    tree_manifest = client.build_tree(
        result.acquisition_manifest_path,
        summarize=True,
        tree_run_id="007-summarized",
    )
    show_json("Tree re-built with LLM summaries", {
        "tree_run_id": tree_manifest.tree_run_id,
        "committed_nodes": tree_manifest.committed_node_count,
    })
else:
    print("No gateway — tree remains deterministic (no LLM summaries).")

[HierarchyStrategySelected] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 tree=007-summarized strategy=inferred_with_llm_assist
[NodeSummarized] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 node=a1147d47018b372a903d9586f5261aa50e6a079e2b4551a3c5f6259309b0f594
[NodeSummarized] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 node=f02d740c21761d7dfcb83cb2c3260a112005cbacc6f36d8251524570547fc11d
[NodeSummarized] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 node=2efb3a0428579fee40bd2f01a14bdc0eac5c56b9a4e8a15fcee785e4a61bc02b
[NodeSummarized] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 node=e86cad24bd88a5004e91d768bd2a2af6e6d25230afaabd52e3ffc26f50318c5e
[NodeSummarized] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 node=893dfc357e3a02dc91aa68eb4602e66f5e94289f70c6eaa9a7d9304a8279c93b
[NodeSummarized] document=798d2f27d45d2

Groq rotation -> slot-01 (gsk_...AnOM)
Groq rotation -> slot-02 (gsk_...xhRt)
Groq rotation -> slot-03 (gsk_...74oG)
Groq rotation -> slot-04 (gsk_...x09a)


[GatewayCallSucceeded]  op=summarize_leaf_node  model=llama-4-scout-17b-16e-instruct  in=519  out=77  latency=663ms
[GatewayCallAttempted]  op=summarize_leaf_node  model=llama-4-scout-17b-16e-instruct  images=(none)  attempt=1/3
[GatewayCallSucceeded]  op=summarize_leaf_node  model=llama-4-scout-17b-16e-instruct  in=516  out=78  latency=688ms
[GatewayCallAttempted]  op=summarize_leaf_node  model=llama-4-scout-17b-16e-instruct  images=(none)  attempt=1/3


Groq rotation -> slot-05 (gsk_...sDUk)
Groq rotation -> slot-01 (gsk_...AnOM)


[GatewayCallSucceeded]  op=summarize_leaf_node  model=llama-4-scout-17b-16e-instruct  in=583  out=150  latency=886ms
[GatewayCallAttempted]  op=summarize_leaf_node  model=llama-4-scout-17b-16e-instruct  images=(none)  attempt=1/3
[GatewayCallSucceeded]  op=summarize_leaf_node  model=llama-4-scout-17b-16e-instruct  in=564  out=165  latency=898ms
[GatewayCallAttempted]  op=summarize_leaf_node  model=llama-4-scout-17b-16e-instruct  images=(none)  attempt=1/3
[GatewayCallSucceeded]  op=summarize_leaf_node  model=llama-4-scout-17b-16e-instruct  in=531  out=81  latency=285ms
[GatewayCallAttempted]  op=summarize_leaf_node  model=llama-4-scout-17b-16e-instruct  images=(none)  attempt=1/3
[GatewayCallSucceeded]  op=summarize_leaf_node  model=llama-4-scout-17b-16e-instruct  in=539  out=124  latency=407ms
[GatewayCallAttempted]  op=summarize_leaf_node  model=llama-4-scout-17b-16e-instruct  images=(none)  attempt=1/3


...

[GatewayCallAttempted]  op=summarize_leaf_node  model=llama-4-scout-17b-16e-instruct  images=(none)  attempt=2/3
[GatewayCallAttempted]  op=summarize_leaf_node  model=llama-4-scout-17b-16e-instruct  images=(none)  attempt=3/3
[GatewayCallAttempted]  op=summarize_leaf_node  model=llama-4-scout-17b-16e-instruct  images=(none)  attempt=2/3


Groq rotation -> slot-01 (gsk_...AnOM)
Groq rotation -> slot-02 (gsk_...xhRt)
Groq rotation -> slot-03 (gsk_...74oG)

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers



[GatewayCallAttempted]  op=summarize_leaf_node  model=llama-4-scout-17b-16e-instruct  images=(none)  attempt=3/3
--- Logging error ---
Traceback (most recent call last):
  File "/home/pruthvi/.local/share/uv/python/cpython-3.12.12-linux-aarch64-gnu/lib/python3.12/logging/__init__.py", line 1163, in emit
    stream.write(msg + self.terminator)
  File "/home/pruthvi/projects/NullVector/.venv/lib/python3.12/site-packages/ipykernel/iostream.py", line 694, in write
    self._schedule_flush()
  File "/home/pruthvi/projects/NullVector/.venv/lib/python3.12/site-packages/ipykernel/iostream.py", line 590, in _schedule_flush
    self.pub_thread.schedule(_schedule_in_thread)
  File "/home/pruthvi/projects/NullVector/.venv/lib/python3.12/site-packages/ipykernel/iostream.py", line 267, in schedule
    self._event_pipe.send(b"")
  File "/home/pruthvi/projects/NullVector/.venv/lib/python3.12/site-packages/zmq/sugar/socket.py", line 698, in send
    return super().send(data, flags=flags, copy=copy, tra

Groq rotation -> slot-04 (gsk_...x09a)Groq rotation -> slot-05 (gsk_...sDUk)


[GatewayCallSucceeded]  op=summarize_leaf_node  model=llama-4-scout-17b-16e-instruct  in=785  out=86  latency=380ms


## Document Description

Generates a document-level summary from the top-level tree nodes. This is reused by
`client.ask()` — when you ask "what is this book about?", NullVector returns the
pre-built description instead of running retrieval.

In [ ]:
description = client.build_description(
    result.acquisition_manifest_path,
    result.tree_manifest_path,
)

show_json("Description built", {
    "document_id": description.document_id,
    "description_run_id": description.description_run_id,
})

## Search

`client.search()` finds relevant sections using BM25-style structural ranking over
the hierarchical tree. Results include page spans for exact citations.

In [ ]:
hits = client.search("sets", ingest_result=result, limit=5)

for i, hit in enumerate(hits, 1):
    unit = hit.unit
    excerpt = (unit.text or "")[:150].replace("\n", " ")
    print(f"[{i}] score={hit.score:.3f} | pages {unit.page_span.start_page}-{unit.page_span.end_page}")
    print(f"    type={unit.unit_type.value} | {excerpt}...")
    print()

## Grounded QA — Three Answer Modes

`client.ask()` classifies the query intent and picks the right strategy:

- **document_summary** — "What is this book about?" → uses the pre-built description, no retrieval
- **focused_lookup** — "What does the Sets section introduce?" → retrieves and synthesizes with page citations
- **low_evidence** — "zebra invoice compliance" → acknowledges the document doesn't contain relevant content

In [ ]:
qa_examples = {
    "document_summary": "What is this book about?",
    "focused_lookup": "What does the Sets section introduce?",
    "low_evidence": "zebra invoice compliance",
}

for label, query in qa_examples.items():
    response = client.ask(query, ingest_result=result)
    print(f"Q: {query}")
    print(f"  mode={response.answer_mode} | strategy={response.answer_strategy}")
    print(f"  answer: {response.answer[:200]}...")
    if response.citations:
        for c in response.citations:
            print(f"  citation: page {c.page_label} — {(c.quote or '')[:80]}")
    print()

## Results Summary

In [ ]:
show_json("Pipeline summary", {
    "document_id": result.document_id,
    "workspace": str(WORKSPACE),
    "gateway_mode": gateway_mode,
    "acquisition_manifest": result.acquisition_manifest_path,
    "tree_manifest": result.tree_manifest_path,
    "retrieval_manifest": result.retrieval_manifest_path,
})

## Notes

- `client.ingest()` combines acquisition, tree build, and retrieval corpus construction into one call
- `client.build_tree(summarize=True)` re-builds the tree with LLM-generated summaries when a gateway is available
- `client.build_description()` is separate because it's optional and can use a live LLM gateway
- `client.search()` and `client.ask()` accept `ingest_result=`, `document_id=`, or `retrieval_manifest_path=`
- All artifacts are persisted in PostgreSQL — omitted client run IDs now generate fresh runs by default
- When `gateway=None` (no LiteLLM), the entire pipeline is deterministic — no LLM calls
- When a live Groq gateway is available, tree summarization and LLM-backed descriptions are enabled
- For service-level APIs (tree search, preference search, compaction), import from `nullvector.retrieval` and `nullvector.tree`
- Set `NULLVECTOR_NOTEBOOK_ENABLE_BRAIN=0` to force deterministic-only mode even when LiteLLM is available